In [76]:
pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# reading files 
import os 
import sys 
sys.path.append(os.path.abspath('..'))
import pandas as pd
from  config import PROJECT_FILE_PATH, BILLIBNG_FILE_PATH


In [ ]:
def load_dataset():
    '''
    Loading dateset making sure file path is correct
    Return : 
        - tuple(df,df)
    '''
    try : 
        billing_data = pd.read_csv(BILLIBNG_FILE_PATH)
        project_data = pd.read_csv(PROJECT_FILE_PATH)
        return billing_data, project_data
    except Exception as e : 
        print(e)

In [ ]:
project_data.head()

,project_id,client_name,services,scope,last_delivery_date,delivery_status
0,P2001,Acme Inc.,Design + Development,Product page redesign,2026-08-05,On track
1,P2002,NIKE,CRO + Design + Development,Shopify theme improvements,2026-07-25,On track
2,P2003,GLOW SKIN,Retention,Analytics and CRO audit,2026-06-20,Active
3,P2004,Adidas AG,CRO,Lifecycle email and SMS,2026-07-30,active
4,P2005,BlueBottle,Retention + Design,Analytics and CRO audit,2026-08-02,On track


In [54]:
# ToDo:
# 1- clean data (removing corrupted records and null)
# 2- Normalize client_name & last_delivery_date & end_date
# make all text lower case / remove any suffix / remove punctuation / remove spaces 

In [6]:
import re
LEGAL_SUFFIX = ['inc','ltd','co','company','llc']

def normalize_brandname(name) : 
    '''
    This function is used to normalize_text, remove_punctuation, removing_suffix and join the name to normalize spaces
    Input : 
          - name : str  client name as
    Return : 
          - Normalized Text. 
    '''
    if name is not None : 
        text = str(name).lower().strip()
        text = re.sub(r'[^a-z]'," ", text)
        text = [r for r in text.split() if r not in LEGAL_SUFFIX]
        text = ''.join(text)
        return text 
    else : 
        return None

In [ ]:
for b in project_data['client_name'] : 
    brandName_Normalize(b)

In [7]:
# we neeed to normalize date
# Concern : is all dates has same timezone or not 
DATE_FORMAT = ['%Y-%m-%d', '%d/%m/%Y', '%d %b %Y']
def normalize_date(date) : 

    for format in DATE_FORMAT : 
        try : 
            return pd.to_datetime(date, format=format)
        except (ValueError, TypeError) : 
            continue
    return pd.NaT
    

In [ ]:
billing_data['end_date'].map(normalize_date)

In [ ]:
!pip install pydantic 

In [47]:
from pydantic import BaseModel, ConfigDict
from typing import Optional
class MatchData(BaseModel) : 
    model_config = ConfigDict(arbitrary_types_allowed=True)
    billing_index : int
    project_index : Optional[int] 
    normalized_name : str 
    last_contract_date : pd.Timestamp
    match_found : bool

In [71]:
! pip install rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 2.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from rapidfuzz import process

In [24]:
billing_data, project_data =  load_dataset()

billing_data.columns, project_data.columns

(Index(['record_id', 'client_name', 'start_date', 'end_date', 'monthly_fee',
        'currency', 'retainer_status'],
       dtype='str'),
 Index(['project_id', 'client_name', 'services', 'scope', 'last_delivery_date',
        'delivery_status'],
       dtype='str'))

In [ ]:
# def main() : 
billing_data, project_data =  load_dataset()

billing_data['client_name'] = billing_data['client_name'].apply(normalize_brandname)
project_data['client_name'] = project_data['client_name'].apply(normalize_brandname)

billing_data['start_date']  = billing_data['start_date'].apply(normalize_date)
billing_data['end_date']    = billing_data['end_date'].apply(normalize_date)
project_data['last_delivery_date'] = project_data['last_delivery_date'].apply(normalize_date)


billing_data = billing_data.drop_duplicates()
project_data = project_data.drop_duplicates()


results =[]
invalid_projects = []
for idx, client_name in enumerate(billing_data['client_name']) : 
   output = process.extractOne(client_name,project_data['client_name'] )

   if output[1] >= 70.00 : 
      data   = MatchData(billing_index=idx,
             project_index=output[-1],
             normalized_name=client_name,
             last_contract_date=max(billing_data.loc[idx, 'end_date'], 
                                    project_data.loc[output[-1],'last_delivery_date']),
            match_found= True)
      billing_data.drop(idx,inplace=True)
      results.append(data)
   else : 
      data = MatchData(billing_index = idx,
             project_index = None ,
             normalized_name=client_name,
             last_contract_date=billing_data.loc[idx, 'end_date'],
             match_found= False)
      results.append(data)

if len(project_data) > 0 :
   for idx, client_name in enumerate(project_data['client_name']) : 
      data = MatchData(billing_index = None,
                   project_index = idx ,
                   normalized_name=client_name,
                   last_contract_date=project_data.loc[idx, 'end_date'],
                   match_found= False)
      invalid_projects.append(data)


('acme', 100.0, 0)


In [46]:
output

('acme', 100.0, 0)